Two gold tables are built from the silver layer that reconciles batch and stream data where batch data runs on a nightly trigger and stream data runs on a 15min trigger for quick analysis.<br>
**Tables used:**<br>
1. silver_orders
2. dim_customers
3. dim_products<br>

since these are aggregates, an **overwrite each run** is the right call

<mark><u>**Gold Tables**</u></mark>:
1. **gold_daily_sales**:
<br> _your batch-cadence view_:<br> revenue, order count, avg order value, units sold, by day and category. Excludes cancelled/refunded orders so <u>**it reflects actual realized sales**</u>. This is what a "yesterday's performance" Power BI page reads from
    
2. **gold_live_orders_last_hour**:
<br>_your speed-layer view_: <br> what's happening in the last 60 minutes, split by whether each order is is_reconciled (confirmed by batch) or not (stream-only, still in flight). This split is deliberate — it lets your BI report show "X confirmed orders + Y pending orders" as two honest numbers instead of quietly guessing.

<u>_Daily sales should represent real revenue; <br>
the live view is about operational visibility (you'd want to see a cancellation spike in the last hour, not hide it)_</u>


In [1]:
from pyspark.sql import functions as F

StatementMeta(, f7bdb4f3-3e59-4d34-b387-6664ab84cd50, 3, Finished, Available, Finished, False)

In [3]:
silver_orders = spark.read.table("silver.silver_orders")
dim_customers = spark.read.table("bronze.dim_customers")
dim_products = spark.read.table("bronze.dim_products")

StatementMeta(, f7bdb4f3-3e59-4d34-b387-6664ab84cd50, 5, Finished, Available, Finished, False)

<u>**Enrich orders with dimension attributes**</u><br>
1. net_revenue accounts for discount; nulls in discount_pct<br>
(stream-only rows not yet confirmed by batch) default to 0.

In [5]:
enriched = (
    silver_orders.alias("o")
    .join(dim_products.alias("p"), on="product_id", how="left")
    .join(dim_customers.alias("c"), on="customer_id", how="left")
    .withColumn("discount_pct_safe", F.coalesce(F.col("discount_pct"), F.lit(0.0)))
    .withColumn(
        "net_revenue",
        F.col("o.quantity") * F.col("o.unit_price") * (F.lit(1) - F.col("discount_pct_safe"))
    )
    .withColumn("order_date", F.to_date(F.col("order_ts")))
    .select(
        "order_id", "order_ts", "order_date", "status", "source_system", "is_reconciled",
        "quantity", "o.unit_price", "discount_pct_safe", "net_revenue",
        F.col("p.category").alias("category"),
        F.col("c.country").alias("country"),
    )
)

StatementMeta(, f7bdb4f3-3e59-4d34-b387-6664ab84cd50, 7, Finished, Available, Finished, False)

In [6]:
gold_daily_sales = (
    enriched
    .filter(~F.col("status").isin("Cancelled", "Refunded"))
    .groupBy("order_date", "category")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("net_revenue").alias("total_revenue"),
        F.avg("net_revenue").alias("avg_order_value"),
        F.sum("quantity").alias("units_sold"),
    )
    .orderBy("order_date", "category")
)
 
gold_daily_sales.write.format("delta").mode("overwrite").saveAsTable("gold.gold_daily_sales")
print(f"gold_daily_sales rows: {gold_daily_sales.count()}")
display(gold_daily_sales.limit(20))

StatementMeta(, f7bdb4f3-3e59-4d34-b387-6664ab84cd50, 8, Finished, Available, Finished, False)

gold_daily_sales rows: 17


SynapseWidget(Synapse.DataFrame, ab07a46c-11ea-4020-a9ed-0140789b9bfa)

_**gold_live_orders_last_hour**
<br>_Speed-layer aggregate_: <br>
what's happening right now, refreshed
on a tight schedule (e.g. every 10-15 min via pipeline).<br>
Includes stream-only orders (is_reconciled = False) on purpose
these are the ones batch hasn't caught up to yet.


In [13]:
max_order_ts = enriched.agg(F.max("order_ts")).collect()[0][0]
one_hour_before_latest = F.lit(max_order_ts) - F.expr("INTERVAL 1 HOURS")

gold_live_orders_last_hour = (
    enriched
    .filter(F.col("order_ts") >= one_hour_before_latest)
    .groupBy("category", "is_reconciled")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("net_revenue").alias("revenue_so_far"),
    )
    .orderBy("category", "is_reconciled")
)

gold_live_orders_last_hour.write.format("delta").mode("overwrite").saveAsTable("gold.gold_live_orders_last_hour")
print(f"gold_live_orders_last_hour rows: {gold_live_orders_last_hour.count()}")
display(gold_live_orders_last_hour)

StatementMeta(, f7bdb4f3-3e59-4d34-b387-6664ab84cd50, 15, Finished, Available, Finished, False)

gold_live_orders_last_hour rows: 12


SynapseWidget(Synapse.DataFrame, d7376f23-b03c-44bd-ba66-0ff7497a4260)

In [14]:
print("Reconciliation split in last-hour window:")
spark.read.table("gold.gold_live_orders_last_hour").groupBy("is_reconciled").agg(
    F.sum("order_count").alias("orders")
).show()

StatementMeta(, f7bdb4f3-3e59-4d34-b387-6664ab84cd50, 16, Finished, Available, Finished, False)

Reconciliation split in last-hour window:
+-------------+------+
|is_reconciled|orders|
+-------------+------+
|         true|    71|
|        false|    81|
+-------------+------+

